# 06 · Conversation memory and context windows

After this notebook you can count the tokens a conversation really sends, keep a chat inside its context window with a sliding window, a token budget or a running summary, see what each one forgets, estimate how the cost grows over many turns, and lay out prompts so provider-side prompt caching actually hits.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · OpenAI API basics](01_openai_api_basics.ipynb) (section 4: the API is stateless). Background: [tokenization](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb)

## Why this matters in interviews

- "The chatbot forgets things after a while. Why?" and "why did our bill triple?" have the same root cause: the API is stateless and every turn resends the history.
- Memory design (window vs summary vs retrieval, and what to pin) is a standard system-design follow-up for assistants and agents.
- Prompt caching is the single biggest cost lever for long conversations and agents, and the classic bug that silently disables it is a favourite trap question.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo05` | What is the context window, and what actually happens when you exceed it? |
| `pd26` | What is the cost of a multi-turn conversation, and why does it surprise people? |
| `pr10` | What are the layers of memory in an LLM application? |
| `pr11` | What is conversation summarisation, and what is the trap? |
| `pr12` | Sliding window versus summary versus retrieval for conversation history — which and why? |
| `in16` | What is context trimming, and what are the strategies? |
| `ag15` | An agent has been running for 50 turns and suddenly forgets everything from the beginning. What happened? |
| `pr30` | What is the "lost in the middle" effect and how do you design around it? |
| `in03` | What is prompt caching and how is it different from a KV cache? |
| `pd19` | How do you keep prompt caching working as your system evolves? |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import tiktoken
    _enc = tiktoken.get_encoding("o200k_base")        # the tokenizer of the gpt-4o / 4.1 / 5 families
    encode = _enc.encode
    COUNTER = "tiktoken o200k_base"
except Exception as e:                                   # no tiktoken, or its vocabulary file can't be downloaded
    encode = lambda s: s.split() + [""] * (len(s) // 16)  # crude stand-in, roughly 4 characters per token
    COUNTER = f"approximation ({type(e).__name__})"

def count_tokens(text):
    return len(encode(text or ""))

def count_messages(messages):
    """Chat-format overhead: ~3 tokens of role/separator markup per message, +3 to prime the reply."""
    return sum(count_tokens(m.get("content")) + 3 for m in messages) + 3

def ask(messages, **kw):
    r = client.chat.completions.create(model=MODEL, messages=messages, **kw)
    return r.choices[0].message.content, r.usage

print("token counter:", COUNTER, "| provider:", PROVIDER, "| model:", MODEL)
print("'Hello, how are you today?' ->", count_tokens("Hello, how are you today?"), "tokens")

## 1. The model remembers nothing; you resend everything

**In plain English:** every API call starts from a blank slate. "Memory" is the list of messages *your code* sends. If a fact isn't in this request, the model doesn't know it, however recently it was said.

We'll use one recorded conversation throughout: a trip-planning chat, with the assistant's replies written out so the demos cost nothing. The user mentions their name only in turn 1.

In [ ]:
SYSTEM = {"role": "system", "content": "You are a friendly travel-planning assistant. Keep answers short."}
SCRIPT = [
    ("Hi! My name is Mamitha. I'm planning a trip to Japan in April.",
     "Lovely, April is cherry-blossom season. How long will you stay, and what do you enjoy?"),
    ("Ten days. I love food and easy hikes, and I don't like crowds.",
     "Then split it between Kyoto, the Kiso Valley trails and a quieter onsen town, and avoid the big viewing parks at weekends."),
    ("My budget is 3000 dollars without flights.",
     "That works: mid-range ryokan, rail passes and good food fit in about 250-300 dollars a day."),
    ("Should I get a rail pass?",
     "For Tokyo-Kyoto plus the Kiso Valley, regional passes usually beat the national one. Price the exact legs first."),
    ("What should I eat in Kyoto?",
     "Try yudofu, kaiseki at lunch when it is cheaper, and the Nishiki market in the morning before the crowds."),
    ("Is the Nakasendo trail hard?",
     "The Magome to Tsumago section is about 8 km on gentle paths, easy for most people in three hours."),
    ("Can I send my luggage ahead?",
     "Yes, takkyubin forwarding services carry bags between hotels overnight for a modest fee."),
    ("What about an onsen with tattoos?",
     "Many ryokan offer private baths, which avoids the tattoo rules of public onsen entirely."),
    ("Do I need cash?",
     "Carry some yen for small shops and rural inns; cards work in cities and at most stations."),
    ("How do I get from Kyoto to the Kiso Valley?",
     "Take the shinkansen to Nagoya, then the limited express to Nakatsugawa and a short bus to Magome."),
]
history = [SYSTEM]
for user, assistant in SCRIPT:
    history += [{"role": "user", "content": user}, {"role": "assistant", "content": assistant}]
QUESTION = {"role": "user", "content": "What is my name?"}

reply_full, usage_full = ask(history + [QUESTION])
reply_none, _ = ask([SYSTEM, QUESTION])
print(f"with the full history ({len(history)} messages): {reply_full}")
print(f"with no history                  : {reply_none}")
assert "Mamitha" in reply_full and "Mamitha" not in reply_none

## 2. Count what you send

Everything is measured in tokens: the context limit, the bill, the latency. Count locally with the model's tokenizer (`tiktoken` for OpenAI models), including the small per-message overhead of the chat format, and compare with what the API reports in `usage.prompt_tokens`:

In [ ]:
local = count_messages(history + [QUESTION])
print(f"local count ({COUNTER}): {local} tokens")
print(f"API usage.prompt_tokens      : {usage_full.prompt_tokens} tokens")
per_turn = [count_messages(history[:1 + 2 * k] + [QUESTION]) for k in range(len(SCRIPT) + 1)]
print("prompt size if asked after turn 0..10:", per_turn)
assert per_turn == sorted(per_turn)

Against OpenAI the two numbers agree to within a few tokens. The offline stand-in only *estimates* usage (characters ÷ 4), so offline any agreement is approximate or coincidental. Count locally anyway: it's how you enforce a budget *before* sending, and how you estimate cost before building. Typical English is about 4 characters or 0.75 words per token. Code, non-Latin scripts and JSON cost more tokens per character.

## 3. Context windows, and what happens when you exceed them

Approximate limits as of 2026. They change often, so check the provider's model page:

| Model family | Context window (input + output) | Notes |
|---|---|---|
| OpenAI `gpt-4.1` / `-mini` / `-nano` | ~1M tokens | output cap ~32k |
| OpenAI `gpt-4o` / `gpt-4o-mini` | 128k | output cap ~16k |
| OpenAI `gpt-5` family | ~400k | includes hidden reasoning tokens in the output budget |
| Anthropic Claude (Sonnet / Opus class) | 200k (1M options on some models) | |
| Google Gemini 2.x / 3 Pro class | ~1M | |
| Llama 3.x, Qwen 2.5 (open weights) | 32k-128k | whatever the model was trained for |
| **Ollama default** | **4k** on machines with < 24 GB VRAM | raise with `OLLAMA_CONTEXT_LENGTH`. Longer prompts are truncated |

Go over the limit and one of two things happens:

1. **An error.** OpenAI returns a 400 (`context_length_exceeded`). Loud, and the good case.
2. **Silent truncation.** Local servers, and many frameworks' "helpful" trimming, drop tokens or messages, usually **the oldest first**. The oldest message is the system prompt.

In [ ]:
def naive_truncate(messages, keep_last):
    return messages[-keep_last:]                        # the one-liner that ships to production

cut = naive_truncate(history + [QUESTION], keep_last=6)
print("roles after naive truncation:", [m["role"] for m in cut])
print("system prompt still there?  ", any(m["role"] == "system" for m in cut))
assert not any(m["role"] == "system" for m in cut)

That's the classic cause of "after 50 turns the agent forgot everything, including its rules" (`ag15`): no error, the persona and constraints just quietly disappear. Rule one of every trimming strategy is to **pin the system prompt**.

## 4. Three trimming strategies

| Strategy | Keeps | Loses | Extra cost |
|---|---|---|---|
| **Sliding window** | the last *N* turns verbatim | everything older, completely | none |
| **Token budget** | as many recent turns as fit in *B* tokens | everything older | none (needs a token counter) |
| **Summary buffer** | a running summary + the last *k* turns | specifics, gradually | a summarisation call now and then |

All three pin the system prompt and cut **whole turns** (never half a user/assistant pair, never an assistant `tool_calls` message without its tool replies, which is a 400: see [notebook 04](04_function_calling.ipynb)).

In [ ]:
def turns(messages):
    """Split into (system, [turn, ...]) where a turn is a user message plus everything up to the next user message."""
    system = [m for m in messages if m["role"] == "system"]
    body, out = [m for m in messages if m["role"] != "system"], []
    for m in body:
        if m["role"] == "user" or not out:
            out.append([])
        out[-1].append(m)
    return system, out

def sliding_window(messages, n_turns):
    system, ts = turns(messages)
    return system + [m for t in ts[-n_turns:] for m in t]

def token_budget(messages, budget):
    system, ts = turns(messages)
    kept, used = [], count_messages(system)
    for t in reversed(ts):                               # newest first, until the budget is spent
        cost = count_messages(t) - 3
        if used + cost > budget:
            break
        kept.insert(0, t)
        used += cost
    return system + [m for t in kept for m in t]

def summary_buffer(messages, keep_turns, summarize):
    system, ts = turns(messages)
    old, recent = ts[:-keep_turns], ts[-keep_turns:]
    if not old:
        return messages
    transcript = "\n".join(f"{m['role']}: {m['content']}" for t in old for m in t)
    summary = summarize(transcript)
    memo = {"role": "system", "content": f"Summary of the earlier conversation: {summary}"}
    return system + [memo] + [m for t in recent for m in t]

def llm_summarize(transcript):
    text, _ = ask([{"role": "user", "content":
                    "Summarize this conversation for your own memory in at most 5 short sentences. "
                    "Keep every fact about the user exactly: name, dates, budget, preferences, decisions.\n\n" + transcript}])
    return text

full = history + [QUESTION]
views = {"full history": full,
         "sliding window (last 3 turns)": sliding_window(full, 3),
         "token budget (250 tokens)": token_budget(full, 250),
         "summary buffer (summary + last 3 turns)": summary_buffer(full, 3, llm_summarize)}
print(views["summary buffer (summary + last 3 turns)"][1]["content"][:300])

Now ask the same question, *"What is my name?"*, with each view of the conversation:

In [ ]:
rows = []
for name, msgs in views.items():
    reply, _ = ask(msgs)
    rows.append({"strategy": name, "messages": len(msgs), "prompt tokens": count_messages(msgs),
                 "system pinned": msgs[0]["role"] == "system", "remembers name": "Mamitha" in reply, "reply": reply[:45]})
result = pd.DataFrame(rows).set_index("strategy")
print(result.to_string())

assert result["system pinned"].all()
assert result.loc["full history", "remembers name"] and result.loc["summary buffer (summary + last 3 turns)", "remembers name"]
assert not result.loc["sliding window (last 3 turns)", "remembers name"]
assert result["prompt tokens"].iloc[1:].max() < result.loc["full history", "prompt tokens"]

The window and the budget are cheap and exact about recent turns, and they forget the name completely: it was in turn 1. The summary buffer remembers it at a fraction of the tokens, *if* the summary kept it. Offline the stand-in's "summary" is just the first two sentences, which happen to contain the name. With a real model it depends on your summarisation prompt, which is why that prompt says "keep every fact about the user exactly".

The traps (`pr11`):

- **Summarise from the original transcript, never from the previous summary.** Summaries of summaries drift: "refund order 4471 for $89.20" becomes "a billing issue".
- **Pin hard facts separately** (ids, names, amounts, decisions) in a structured "profile" block that's never rewritten (exercise 2).
- **Trigger on a token threshold, not every turn**, and summarise in the background between turns, not on the critical path.
- For long-lived assistants, add **retrieval over the archived transcript** ("what did I say about hotels in March?"). The best systems combine pinned facts + summary + last N turns + retrieval, each with an explicit token budget (`pr12`).

## 5. The token budget: where the window goes

**In plain English:** the context window is shared by everything. The system prompt, retrieved documents, the conversation, and **room for the answer** (the output is part of the window too). Here's a 16k-token model in a chatbot that also retrieves documents, with about 350 tokens per turn:

In [ ]:
WINDOW = 16_384
SYSTEM_T, RETRIEVED_T, OUTPUT_RESERVE_T, PER_TURN_T, SUMMARY_T = 800, 3_000, 1_000, 350, 400
layouts = {
    "turn 5\nfull history": [SYSTEM_T, RETRIEVED_T, 4 * PER_TURN_T, OUTPUT_RESERVE_T],
    "turn 40\nfull history": [SYSTEM_T, RETRIEVED_T, 39 * PER_TURN_T, OUTPUT_RESERVE_T],
    "turn 40\nlast 6 turns": [SYSTEM_T, RETRIEVED_T, 6 * PER_TURN_T, OUTPUT_RESERVE_T],
    "turn 40\nsummary + last 4": [SYSTEM_T, RETRIEVED_T, SUMMARY_T + 4 * PER_TURN_T, OUTPUT_RESERVE_T],
}
parts = ["system prompt", "retrieved context", "conversation history", "output reserve"]
colors = ["#495057", "#1c7ed6", "#e8590c", "#2f9e44"]
fig, ax = plt.subplots(figsize=(9, 4))
bottom = np.zeros(len(layouts))
vals = np.array(list(layouts.values()))
for j, (part, c) in enumerate(zip(parts, colors)):
    ax.bar(list(layouts), vals[:, j], bottom=bottom, color=c, label=part)
    bottom += vals[:, j]
ax.axhline(WINDOW, color="red", ls="--", lw=1.5)
ax.text(-0.4, WINDOW + 300, "16k context window", color="red", ha="left", fontsize=9)
ax.set_ylabel("tokens")
ax.set_title("Everything shares one window, including the answer")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
ax.set_ylim(0, vals.sum(axis=1).max() * 1.12)
plt.show()
totals = dict(zip([k.replace("\n", " ") for k in layouts], vals.sum(axis=1)))
print({k: int(v) for k, v in totals.items()})
assert totals["turn 40 full history"] > WINDOW > totals["turn 40 last 6 turns"]

Two rules fall out of the chart. **Reserve the output budget explicitly**: if history eats it, the answer is truncated mid-sentence (`finish_reason="length"`), which users read as a broken model. And **give each part a budget** (e.g. history ≤ 4k, retrieval ≤ 6k), so one part can't crowd out the others.

## 6. What long conversations cost

With a stateless API, turn *k* resends turns 1…k-1, so input tokens per call grow linearly and the **total grows with the square** of the number of turns (`pd26`). Here's a simulation over 40 turns at `gpt-4.1-mini` prices (as of Sept 2026: $0.40 per 1M input tokens, $1.60 per 1M output):

In [ ]:
PRICE_IN, PRICE_OUT = 0.40 / 1e6, 1.60 / 1e6
USER_T, REPLY_T, SYS_T = 60, 190, 800
N = 40

def simulate(strategy, window_turns=6, summary_every=10, summary_t=400, cached_discount=None):
    """Cumulative USD after each turn. cached_discount: share of the price saved on the reused prefix."""
    total, curve, summary_calls = 0.0, [], 0
    for k in range(1, N + 1):
        if strategy == "full":
            hist = (k - 1) * (USER_T + REPLY_T)
        elif strategy == "window":
            hist = min(k - 1, window_turns) * (USER_T + REPLY_T)
        else:                                            # summary buffer: summary + turns since the last summary
            if k > 1 and (k - 1) % summary_every == 0:
                total += (summary_every * (USER_T + REPLY_T) + summary_t) * PRICE_IN + summary_t * PRICE_OUT
                summary_calls += 1
            hist = (summary_t if k > summary_every else 0) + ((k - 1) % summary_every) * (USER_T + REPLY_T)
        prompt = SYS_T + hist + USER_T
        if cached_discount and k > 1 and prompt - USER_T - REPLY_T >= 1024:
            reused = prompt - USER_T - REPLY_T           # everything up to the previous turn was sent before
            total += reused * PRICE_IN * (1 - cached_discount) + (prompt - reused) * PRICE_IN + REPLY_T * PRICE_OUT
        else:
            total += prompt * PRICE_IN + REPLY_T * PRICE_OUT
        curve.append(total)
    return np.array(curve)

curves = {"full history": simulate("full"),
          "full history + prompt caching (assumed 75% off cached input)": simulate("full", cached_discount=0.75),
          "sliding window (6 turns)": simulate("window"),
          "summary buffer (every 10 turns)": simulate("summary")}
fig, ax = plt.subplots(figsize=(9, 4))
for (name, c), ls in zip(curves.items(), ["-", ":", "-", "-"]):
    ax.plot(range(1, N + 1), c * 1000, ls=ls, lw=2, label=name)
ax.set_xlabel("turn")
ax.set_ylabel("cumulative cost (USD per 1,000 conversations)")
ax.set_title("Resending full history grows quadratically; trimming keeps it linear")
ax.legend(fontsize=8)
plt.show()
final = {k: v[-1] * 1000 for k, v in curves.items()}
for k, v in final.items():
    print(f"${v:6.2f} per 1,000 conversations of 40 turns  <- {k}")
assert final["full history"] > final["summary buffer (every 10 turns)"] and final["full history"] > final["sliding window (6 turns)"]

The full-history curve bends upward: turn 40 costs several times turn 1. Windowing and summaries straighten it. Caching flattens it too, *without forgetting anything*, because each call's growing prefix is exactly what was sent last time. The 75% cached discount is an assumption for illustration: discounts differ by model and change, so check the pricing page. In agent loops (many calls per task, same long prefix) caching is usually the biggest single saving.

## 7. "Lost in the middle"

**In plain English:** models find information at the start and at the end of a long prompt more reliably than information buried in the middle. Accuracy by position is roughly U-shaped (Liu et al., 2023, "Lost in the Middle"), and the dip deepens as contexts get longer. Newer long-context models are better at this, but not immune. A big window is not the same as uniform attention over it.

| Design rule | Why |
|---|---|
| Put the most relevant retrieved chunk **first** (rerank), not in position 7 of 20 | it lands where attention is strongest |
| **Restate the task and key constraints at the end**, right before the question | the end is the other strong position |
| Retrieve a *modest* number of chunks rather than "as many as fit" | more text means a deeper middle and higher cost |
| For huge documents: extract relevant passages first, then answer over the extract | two short prompts beat one enormous one |
| Test *your* model at *your* context length with a needle-in-a-haystack eval | the effect varies by model and length |

## 8. Provider prompt caching: put stable content first

**In plain English:** if many requests start with the same long text, the provider can keep its already-computed internal state (the KV cache of that prefix) and skip re-reading it. You pay less and get the first token sooner. OpenAI does this automatically for prompts of **≥ 1,024 tokens**, in 128-token increments, and reports it in `usage.prompt_tokens_details.cached_tokens` (Responses API: `usage.input_tokens_details.cached_tokens`). Caches live for minutes of inactivity. Discounts and details vary by model, so check the docs.

It only works on an **identical prefix**. One changed character near the top (a timestamp, a user name, a shuffled tool list) invalidates everything after it. Compare two layouts of the same content, for two consecutive requests:

In [ ]:
POLICY = "\n".join(f"Rule {i}: " + "Guests may cancel free of charge up to 48 hours before check-in; after that the first night is billed. " * 2
                   for i in range(1, 31))                      # ~1,300 tokens of stable instructions

def build(layout, now, question):
    if layout == "stable first":
        return [{"role": "system", "content": POLICY}, {"role": "user", "content": f"(sent {now}) {question}"}]
    return [{"role": "system", "content": f"Current time: {now}\n" + POLICY}, {"role": "user", "content": question}]

def cacheable_prefix(msgs_a, msgs_b):
    a, b = encode("".join(m["content"] for m in msgs_a)), encode("".join(m["content"] for m in msgs_b))
    shared = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
    return shared, (shared // 128) * 128 if shared >= 1024 else 0

rows = []
for layout in ["stable first", "timestamp first"]:
    first = build(layout, "2026-09-29 10:00:01", "Can I cancel tomorrow's booking for free?")
    second = build(layout, "2026-09-29 10:00:07", "What if I cancel on arrival day?")
    shared, cacheable = cacheable_prefix(first, second)
    rows.append({"layout": layout, "prompt tokens": count_messages(second), "shared prefix": shared,
                 "cacheable (>=1024, x128)": cacheable})
cache_table = pd.DataFrame(rows).set_index("layout")
print(cache_table.to_string())
assert cache_table.loc["stable first", "cacheable (>=1024, x128)"] >= 1024
assert cache_table.loc["timestamp first", "cacheable (>=1024, x128)"] == 0

Same information, same length, and one layout can reuse ~1,200 tokens per request while the other reuses nothing, because the timestamp in line one differs every time. Now send the good layout twice for real and look at what the provider reports:

In [ ]:
for i, q in enumerate(["Can I cancel tomorrow's booking for free?", "What if I cancel on arrival day?"], 1):
    r = client.chat.completions.create(model=MODEL, messages=build("stable first", f"2026-09-29 10:00:0{i}", q),
                                       max_tokens=40)
    details = r.usage.prompt_tokens_details
    print(f"call {i}: prompt_tokens={r.usage.prompt_tokens}, cached_tokens={details.cached_tokens if details else 'n/a'}")

With OpenAI the second call typically reports `cached_tokens` of 1,024 or more. It's best effort: requests can land on a machine without the cache, and a cold first call caches nothing. The offline stand-in has no cache and always reports 0. Treat the cache-hit rate as a metric and alert when it drops (`pd19`). Keep caching working like this:

- **Stable first:** system prompt → tool schemas (in a *fixed* order) → few-shot examples → long documents.
- **Volatile last:** the user's message, timestamps, request ids, per-turn retrieved chunks.
- Batch prompt edits: every change to the prefix invalidates the cache for all users.

**Prompt cache vs KV cache (`in03`):** the KV cache lives *inside one generation*, reusing attention keys/values for the tokens generated so far ([KV cache notebook](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb)). The prompt cache keeps the KV state of a *prefix across requests*. A semantic cache (your own) skips the call entirely for similar questions, and it's the only one of the three that can return a wrong answer.

## Try it yourself

**1. Never split a tool round.** Here's a history containing a tool call. Show that `messages[-3:]` produces a list that starts with an orphan `tool` message (the API rejects it), while `sliding_window(..., 2)` from above keeps whole turns and is accepted.

In [ ]:
# Solution — try it yourself first, then run this
import openai
call = {"id": "call_1", "type": "function", "function": {"name": "get_weather", "arguments": '{"city": "Kyoto"}'}}
tool_history = [SYSTEM,
                {"role": "user", "content": "Weather in Kyoto?"},
                {"role": "assistant", "content": None, "tool_calls": [call]},
                {"role": "tool", "tool_call_id": "call_1", "content": '{"temp_c": 17}'},
                {"role": "assistant", "content": "It is 17 C in Kyoto."},
                {"role": "user", "content": "And what is my name?"}]
TOOLS = [{"type": "function", "function": {"name": "get_weather", "parameters": {"type": "object", "properties": {"city": {"type": "string"}}}}}]
for label, msgs in [("messages[-3:]", tool_history[-3:]), ("sliding_window(2)", sliding_window(tool_history, 2))]:
    try:
        client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS)
        print(f"{label:<18} accepted | roles: {[m['role'] for m in msgs]}")
    except openai.BadRequestError as e:
        print(f"{label:<18} 400 -> {str(e)[:90]}...")
assert sliding_window(tool_history, 2)[1]["role"] == "user"

**2. Pinned facts.** Write `extract_facts(messages)` that pulls "my X is Y" statements out of user turns into a dict (a regex is fine here; in production a small model call does it). Inject the facts as a pinned system block, *in addition to* the 3-turn sliding window, and ask "What is my name?" again. It should now be remembered, for a couple of dozen extra tokens.

In [ ]:
# Solution — try it yourself first, then run this
def extract_facts(messages):
    facts = {}
    for m in messages:
        if m["role"] == "user":
            for key, value in re.findall(r"\bmy (\w+) is ([\w ]+?)(?:[.,!]|$)", m["content"], flags=re.I):
                facts[key.lower()] = value.strip()                 # later statements overwrite earlier ones
    return facts

facts = extract_facts(history)
pinned = {"role": "system", "content": "Known facts about the user: " + "; ".join(f"my {k} is {v}" for k, v in facts.items())}
windowed = sliding_window(full, 3)
with_facts = [windowed[0], pinned] + windowed[1:]
reply, _ = ask(with_facts)
print("facts:", facts)
print(f"window only: {count_messages(windowed)} tokens | window + facts: {count_messages(with_facts)} tokens -> {reply}")
assert "Mamitha" in reply

**3. When does it overflow?** With 800 system tokens, 3,000 tokens of retrieved context, a 1,000-token output reserve and 350 tokens per turn, after how many turns does full history overflow a 4k context (Ollama's default), a 16k one and a 128k one?

In [ ]:
# Solution — try it yourself first, then run this
def overflow_turn(window, fixed=800 + 3_000 + 1_000, per_turn=350):
    return next(k for k in range(1, 10_000) if fixed + k * per_turn > window)

for w in [4_096, 16_384, 128_000]:
    print(f"{w:>7,}-token window overflows at turn {overflow_turn(w)}")
assert overflow_turn(4_096) == 1          # the fixed parts alone don't fit in 4k: this is why defaults matter

## Say it in an interview

- **Statelessness (30 s):** "The API remembers nothing. Every call carries the full context, so input grows linearly per turn and total cost quadratically over a conversation. Memory is a design problem on my side: what to keep verbatim, what to summarise, what to pin, what to retrieve, and a token budget for each, including room for the answer."
- **Strategies:** sliding window (free, forgets old turns completely); token budget (the same, but exact in tokens); summary buffer (keeps the gist, loses specifics, costs a call); pinned facts (names, ids, decisions, never rewritten); retrieval over the transcript for long-lived assistants. Real systems combine them. Always pin the system prompt, and trim whole turns and tool rounds.
- **Exceeding the window:** a 400 if you're lucky, silent truncation of the *oldest* messages (the system prompt!) if you're not. Local servers default to small contexts (Ollama: 4k).
- **Lost in the middle:** attention is strongest at the start and end, so rerank the best chunk first, restate instructions at the end, and keep context modest.
- **Prompt caching:** automatic from 1,024 tokens on OpenAI; an identical prefix is required; stable content first, volatile last; watch `cached_tokens`. The classic bug is a timestamp at the top of the system prompt, which drops the hit rate to zero.
- **Traps:** summarising the summary (drift); "a 1M window means no memory design" (cost, latency, lost-in-the-middle, and everything in the window is visible to the model); `previous_response_id` doesn't make history free.

## Next

- [Agent memory, short and long term](../15_agentic_ai/04_agent_memory_short_and_long_term.ipynb) · [LangGraph checkpointing and memory](../14_langgraph/03_checkpointing_memory_and_time_travel.ipynb)
- [Caching: exact and semantic](../16_production/03_caching_exact_and_semantic.ipynb) · [KV cache and inference speed](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb)
- [RAG pipeline from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb): retrieval as the long-term memory layer
- Theory: [genai_flow course](../../genai_flow/index.html) · [ai_system_design_concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)